In [218]:
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer

In [219]:
# import the data
path = r'E:\Imarticus_ Learning\Jobs - 2026\Models\data\unzipped'
df = pd.read_csv(path + r'\Life Expectancy Data.csv')
df.head()

,Country,Year,Status,Life expectancy,Adult Mortality,infant deaths,Alcohol,percentage expenditure,Hepatitis B,Measles,...,Polio,Total expenditure,Diphtheria,HIV/AIDS,GDP,Population,thinness 1-19 years,thinness 5-9 years,Income composition of resources,Schooling
0,Afghanistan,2015,Developing,65.0,263.0,62,0.01,71.279624,65.0,1154,...,6.0,8.16,65.0,0.1,584.259210,33736494.0,17.2,17.3,0.479,10.1
1,Afghanistan,2014,Developing,59.9,271.0,64,0.01,73.523582,62.0,492,...,58.0,8.18,62.0,0.1,612.696514,327582.0,17.5,17.5,0.476,10.0
2,Afghanistan,2013,Developing,59.9,268.0,66,0.01,73.219243,64.0,430,...,62.0,8.13,64.0,0.1,631.744976,31731688.0,17.7,17.7,0.470,9.9
3,Afghanistan,2012,Developing,59.5,272.0,69,0.01,78.184215,67.0,2787,...,67.0,8.52,67.0,0.1,669.959000,3696958.0,17.9,18.0,0.463,9.8
4,Afghanistan,2011,Developing,59.2,275.0,71,0.01,7.097109,68.0,3013,...,68.0,7.87,68.0,0.1,63.537231,2978599.0,18.2,18.2,0.454,9.5


In [220]:
df.isnull().sum().sort_values(ascending=False)

Population                         652
Hepatitis B                        553
GDP                                448
Total expenditure                  226
Alcohol                            194
Income composition of resources    167
Schooling                          163
 thinness 5-9 years                 34
 thinness  1-19 years               34
 BMI                                34
Polio                               19
Diphtheria                          19
Life expectancy                     10
Adult Mortality                     10
 HIV/AIDS                            0
Country                              0
Year                                 0
Measles                              0
percentage expenditure               0
infant deaths                        0
Status                               0
under-five deaths                    0
dtype: int64

In [221]:
df.isnull().sum().sort_values(ascending=False)

Population                         652
Hepatitis B                        553
GDP                                448
Total expenditure                  226
Alcohol                            194
Income composition of resources    167
Schooling                          163
 thinness 5-9 years                 34
 thinness  1-19 years               34
 BMI                                34
Polio                               19
Diphtheria                          19
Life expectancy                     10
Adult Mortality                     10
 HIV/AIDS                            0
Country                              0
Year                                 0
Measles                              0
percentage expenditure               0
infant deaths                        0
Status                               0
under-five deaths                    0
dtype: int64

In [222]:
# We are droping any null values in Life Expectency column , if treated lead to data leakage
df = df[df['Life expectancy '].notnull()]

In [223]:
df.columns

Index(['Country', 'Year', 'Status', 'Life expectancy ', 'Adult Mortality',
       'infant deaths', 'Alcohol', 'percentage expenditure', 'Hepatitis B',
       'Measles ', ' BMI ', 'under-five deaths ', 'Polio', 'Total expenditure',
       'Diphtheria ', ' HIV/AIDS', 'GDP', 'Population',
       ' thinness  1-19 years', ' thinness 5-9 years',
       'Income composition of resources', 'Schooling'],
      dtype='object')

In [224]:
# columns to be dropped
cols = [#'Country', 'Year', 'Status', 'Life expectancy ', 'Adult Mortality',
       #'infant deaths', 'Alcohol', 'percentage expenditure', 'Hepatitis B',
       #'Measles ', ' BMI ', 
       'under-five deaths ', 
       #'Polio', 'Total expenditure',
       #'Diphtheria ', ' HIV/AIDS',
       'GDP','Population', ' thinness  1-19 years',
       #' thinness 5-9 years', 
       'Income composition of resources', 
       #'Schooling'
       ]
df = df.drop(columns=cols)

# these columns wer droped due to multi colinearity problem

In [225]:
# Hepatitis B
# creating a column named HepB_missing which give information to the model 
df['Hepatitis B_missing'] = df['Hepatitis B'].isna().astype(int)
#Fill the missing values with year wise impuation
df['Hepatitis B'] = df.groupby('Year')['Hepatitis B'].transform(lambda x: x.fillna(x.median()))
df['Hepatitis B'] = df['Hepatitis B'].fillna(df['Hepatitis B'].median())


In [226]:
# Total Expenditure

def fill_expenditure(x):
    if x.isna().sum() == 1:
        return x.ffill()

df['Total expenditure'] = df.groupby('Country')['Total expenditure'].transform(fill_expenditure)

# global fallback for any group where even this didn't resolve everything
df['Total expenditure'] = df['Total expenditure'].fillna(df['Total expenditure'].median())

print(df['Total expenditure'].isna().sum())  # confirm 0

0


In [227]:
# ALCOHOL
mask = (df['Alcohol'] == 0.01) & (df['Year'] >= 2012)
df.loc[mask, 'Alcohol'] = np.nan

df['Alcohol'] = df.groupby('Country')['Alcohol'].transform(lambda x: x.ffill().bfill())

df['Alcohol'] = df.groupby(['Status', 'Year'])['Alcohol'].transform(lambda x: x.fillna(x.median())) # for South Sudan


In [228]:
# SCHOOLING

df['Schooling'] = df.groupby(['Status', 'Year'])['Schooling'].transform(
    lambda x: x.fillna(x.median())
)
df['Schooling'] = df.groupby('Status')['Schooling'].transform(
    lambda x: x.fillna(x.median())
)

In [229]:
# Diphtheria 
df['Diphtheria '] = df.groupby('Country')['Diphtheria '].transform(
    lambda x: x.interpolate(method='linear', limit_direction='both')
)

In [230]:
df.head()

,Country,Year,Status,Life expectancy,Adult Mortality,infant deaths,Alcohol,percentage expenditure,Hepatitis B,Measles,BMI,Polio,Total expenditure,Diphtheria,HIV/AIDS,thinness 5-9 years,Schooling,Hepatitis B_missing
0,Afghanistan,2015,Developing,65.0,263.0,62,0.01,71.279624,65.0,1154,19.1,6.0,5.73,65.0,0.1,17.3,10.1,0
1,Afghanistan,2014,Developing,59.9,271.0,64,0.01,73.523582,62.0,492,18.6,58.0,5.73,62.0,0.1,17.5,10.0,0
2,Afghanistan,2013,Developing,59.9,268.0,66,0.01,73.219243,64.0,430,18.1,62.0,5.73,64.0,0.1,17.7,9.9,0
3,Afghanistan,2012,Developing,59.5,272.0,69,0.01,78.184215,67.0,2787,17.6,67.0,5.73,67.0,0.1,18.0,9.8,0
4,Afghanistan,2011,Developing,59.2,275.0,71,0.01,7.097109,68.0,3013,17.2,68.0,5.73,68.0,0.1,18.2,9.5,0


In [231]:
X = df.drop(columns=['Life expectancy ', 'Country'])
y = df['Life expectancy ']
country = df['Country']

X_train, X_test, y_train, y_test, country_train, country_test = train_test_split(
    X, y, country, test_size=0.2, random_state=42
)

X_train = X_train.copy()
X_test = X_test.copy()

label_encod = LabelEncoder()
X_train['Status'] = label_encod.fit_transform(X_train['Status'])
X_test['Status'] = label_encod.transform(X_test['Status'])

imputer = IterativeImputer(max_iter=10, random_state=0)
X_train_imputed = imputer.fit_transform(X_train)
X_test_imputed = imputer.transform(X_test)

X_train_imputed = pd.DataFrame(X_train_imputed, columns=X_train.columns, index=X_train.index)
X_test_imputed = pd.DataFrame(X_test_imputed, columns=X_test.columns, index=X_test.index)

X_train_imputed.insert(0, 'Country', country_train.values)
X_test_imputed.insert(0, 'Country', country_test.values)

In [232]:
X_train_imputed.head()

,Country,Year,Status,Adult Mortality,infant deaths,Alcohol,percentage expenditure,Hepatitis B,Measles,BMI,Polio,Total expenditure,Diphtheria,HIV/AIDS,thinness 5-9 years,Schooling,Hepatitis B_missing
2268,Serbia,2012.0,1.0,126.0,1.0,9.38,742.510971,97.0,0.0,58.3,93.0,9.89,91.0,0.1,2.1,14.0,0.0
1680,Mauritius,2002.0,1.0,179.0,0.0,4.14,369.631710,88.0,0.0,26.4,88.0,4.24,88.0,0.1,7.9,12.5,0.0
2785,United Republic of Tanzania,2008.0,1.0,376.0,92.0,3.44,0.000000,86.0,3413.0,19.6,89.0,4.21,86.0,7.4,7.3,11.9,0.0
2512,Sweden,2008.0,0.0,62.0,0.0,6.90,8105.590882,92.0,25.0,56.5,98.0,9.23,98.0,0.1,1.3,15.7,1.0
1090,Guinea-Bissau,2015.0,1.0,275.0,4.0,3.57,0.000000,87.0,153.0,26.3,87.0,5.73,87.0,3.2,7.0,9.2,0.0


In [233]:
X_test_imputed.head()

,Country,Year,Status,Adult Mortality,infant deaths,Alcohol,percentage expenditure,Hepatitis B,Measles,BMI,Polio,Total expenditure,Diphtheria,HIV/AIDS,thinness 5-9 years,Schooling,Hepatitis B_missing
2399,South Africa,2009.0,1.0,449.0,46.0,7.60,782.598714,74.0,5857.0,46.4,75.0,8.39,76.0,19.0,9.8,12.8,0.0
196,Bangladesh,2011.0,1.0,14.0,118.0,0.01,62.349885,96.0,5625.0,15.8,96.0,3.16,96.0,0.1,19.2,9.4,0.0
2316,Singapore,2012.0,0.0,59.0,0.0,1.89,6041.858981,97.0,42.0,32.4,97.0,4.22,97.0,0.1,2.1,15.4,0.0
1735,Montenegro,2012.0,1.0,11.0,0.0,6.56,648.133178,9.0,0.0,6.2,94.0,7.25,94.0,0.1,1.9,15.1,0.0
1102,Guinea-Bissau,2003.0,1.0,38.0,5.0,2.16,2.527115,86.0,1158.0,19.0,65.0,5.62,6.0,4.6,9.5,7.4,1.0


In [234]:
# Save
# Existing folder
out_dir = Path(r"E:\Imarticus_ Learning\Jobs - 2026\Models\data\Processed")

# Save files
X_train_imputed.to_csv(out_dir / "X_train.csv", index=False)
X_test_imputed.to_csv(out_dir / "X_test.csv", index=False)
y_train.to_csv(out_dir / "y_train.csv", index=False)
y_test.to_csv(out_dir / "y_test.csv", index=False)
